

> This notebook contains the code to create the dev and holdout split for RQ5 - posthoc LLM based idiomatic Java refactoring - and the test framework to measure the functional and maintainability scores on the refactored code.



In [ ]:
This script uses a fixed random seed of 42 to generate 70/30 split of the test-passing translations from COBOL4J. It also extracts the COBOL source files from the CodeNet dataset
corresponding to these translations

In [ ]:
import json
import random
import os
import shutil
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_FILE = 'Functional/jsons_functional_testing/nonllm_result.json'
TRANSLATIONS_DIR   = 'Functional/COBOL4J_results'
CODENET_DIR        = 'CodeNet_Dataset'
OUTPUT_DIR         = 'RQ5'

RANDOM_SEED        = 42
DEV_RATIO          = 0.7

# ============================================================

def setup_dirs():
    for d in ['dev', 'holdout', 'cobol']:
        os.makedirs(os.path.join(OUTPUT_DIR, d), exist_ok=True)

def extract_passing_pids(results_file):
    with open(results_file, 'r') as f:
        data = json.load(f)
    passing = [
        r['problem_id']
        for r in data['detailed_results']
        if r['overall_status'] == 'passed'
    ]
    print(f"Total passing: {len(passing)}")
    return passing

def split_pids(passing_pids):
    random.seed(RANDOM_SEED)
    shuffled  = random.sample(passing_pids, len(passing_pids))
    split_idx = round(len(shuffled) * DEV_RATIO)
    dev       = shuffled[:split_idx]
    holdout   = shuffled[split_idx:]
    print(f"Dev: {len(dev)} | Holdout: {len(holdout)}")
    return dev, holdout

def copy_java_files(pids, split_name):
    """Copy pid/java folder into rq5/dev or rq5/holdout"""
    copied  = 0
    missing = 0
    for pid in pids:
        src = os.path.join(TRANSLATIONS_DIR, pid)
        dst = os.path.join(OUTPUT_DIR, split_name, pid)
        if not os.path.isdir(src):
            print(f"  [WARN] No translation folder for {pid}")
            missing += 1
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        copied += 1
    print(f"  Copied {copied} folders to {split_name}/ ({missing} missing)")

def copy_cobol_files(pids):
    """Copy COBOL source, input.txt and output.txt into rq5/cobol/pid/"""
    copied  = 0
    missing = 0
    for pid in pids:
        pid_src_dir  = os.path.join(CODENET_DIR, pid)
        cobol_src    = os.path.join(pid_src_dir, 'COBOL')
        pid_out_dir  = os.path.join(OUTPUT_DIR, 'cobol', pid)
        os.makedirs(pid_out_dir, exist_ok=True)

        # Copy first COBOL file alphabetically
        if os.path.isdir(cobol_src):
            cobol_files = sorted([
                f for f in os.listdir(cobol_src)
                if f.endswith(('.cob', '.cbl', '.COB', '.CBL'))
            ])
            if cobol_files:
                shutil.copy2(
                    os.path.join(cobol_src, cobol_files[0]),
                    os.path.join(pid_out_dir, cobol_files[0])
                )
            else:
                print(f"  [WARN] No COBOL files for {pid}")
                missing += 1
                continue
        else:
            print(f"  [WARN] No COBOL dir for {pid}")
            missing += 1
            continue

        # Copy input.txt and output.txt
        for fname in ['input.txt', 'output.txt']:
            src = os.path.join(pid_src_dir, fname)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(pid_out_dir, fname))
            else:
                print(f"  [WARN] {fname} not found for {pid}")

        copied += 1

    print(f"  Copied {copied} pid folders to cobol/ ({missing} missing)")


def save_split(dev, holdout):
    split_path = os.path.join(OUTPUT_DIR, 'rq5_split.json')
    with open(split_path, 'w') as f:
        json.dump({
            'total_passing': len(dev) + len(holdout),
            'dev':           dev,
            'holdout':       holdout,
            'split_ratio':   f"{DEV_RATIO}/{round(1-DEV_RATIO, 1)}",
            'random_seed':   RANDOM_SEED,
        }, f, indent=2)
    print(f"  Split saved to: {split_path}")

def main():
    print("="*60)
    print("RQ5 DATA PREPARATION")
    print("="*60)

    setup_dirs()

    passing_pids    = extract_passing_pids(RESULTS_FILE)
    dev, holdout    = split_pids(passing_pids)
    all_pids        = dev + holdout

    print("\n>> Copying Java translations...")
    copy_java_files(dev,     'dev')
    copy_java_files(holdout, 'holdout')

    print("\n>> Copying COBOL source files...")
    copy_cobol_files(all_pids)

    print("\n>> Saving split...")
    save_split(dev, holdout)

    print("\n>> Done!")
    print(f"   rq5/dev/          — {len(dev)} pid folders")
    print(f"   rq5/holdout/      — {len(holdout)} pid folders")
    print(f"   rq5/cobol/        — {len(all_pids)} .cob files")
    print(f"   rq5/rq5_split.json")

if __name__ == "__main__":
    main()

In [ ]:
import os
import subprocess

# ===== INSTALL OPENSOURCECOBOL4J =====

print("="*80)
print("INSTALLING OPENSOURCECOBOL4J")
print("="*80)

# Install dependencies
print("\n📦 Installing dependencies...")
os.system("apt-get update -qq")
os.system("apt-get install -y -qq default-jdk build-essential bison flex gettext texinfo libgmp-dev autoconf")

# Download and install opensourcecobol4j
print("\n📥 Downloading opensourcecobol4j v1.1.16...")
os.system("curl -L -o opensourcecobol4j-v1.1.16.tar.gz https://github.com/opensourcecobol/opensourcecobol4j/archive/refs/tags/v1.1.16.tar.gz")

print("\n📂 Extracting...")
os.system("tar zxf opensourcecobol4j-v1.1.16.tar.gz")

print("\n🔧 Configuring and building...")
os.chdir("opensourcecobol4j-1.1.16")
os.system("./configure --prefix=/usr/")
os.system("make")
os.system("make install")

# Set CLASSPATH
print("\n⚙️ Setting CLASSPATH...")
os.environ['CLASSPATH'] = '/usr/lib/opensourcecobol4j/libcobj.jar'

# Return to original directory
os.chdir("/content")

print("\n✅ OpenSourceCOBOL4J installed successfully!")

# Verify installation
result = subprocess.run(['cobj', '--version'], capture_output=True, text=True)
print(f"\n{result.stdout}")



In [ ]:
This script contains the test framwork to check test pass rate and maintainability scores using Lizard on the refactored code.

In [ ]:
import os
import json
import subprocess
import shutil
import re
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

# Input — refactored Java files from strategy 1
S1_CLEANED_DIR = 'RQ5/RQ5_results/rq5_strategy1_results/cleaned'

# Test data
CODENET_DIR    = 'CodeNet_Dataset'  # pid/input.txt, pid/output.txt

# Output
OUTPUT_DIR     = 'RQ5/Test_results'
OUTPUT_FILE    = os.path.join(OUTPUT_DIR, 'rq5_s1_evaluation.json')

LIBCOBJ_JAR    = '/usr/lib/opensourcecobol4j/libcobj.jar'
TEMP_COMPILE_DIR = '/content/temp_compile_rq5'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

print(">> Installing lizard...")
os.system("pip install -q lizard")

# ============================================================
# HELPERS
# ============================================================

def extract_main_class_name(java_file_path):
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        m = re.search(r'public\s+class\s+(\w+)', content)
        if m:
            return m.group(1)
        m = re.search(r'class\s+(\w+)', content)
        return m.group(1) if m else None
    except Exception:
        return None

def clean_temp():
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
    os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

def compile_java(java_file_path):
    try:
        class_name = extract_main_class_name(java_file_path)
        if not class_name:
            return False, "Could not extract class name", None
        dest = os.path.join(TEMP_COMPILE_DIR, f"{class_name}.java")
        shutil.copy(java_file_path, dest)
        result = subprocess.run(
            ['javac', '-cp', LIBCOBJ_JAR, dest],
            capture_output=True, text=True, timeout=10, cwd=TEMP_COMPILE_DIR
        )
        if result.returncode == 0:
            return True, None, class_name
        return False, result.stderr.strip(), None
    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None

def run_java_with_input(class_name, input_data):
    try:
        import time
        start = time.time()
        result = subprocess.run(
            ['java', '-cp', f'.:{LIBCOBJ_JAR}', class_name],
            input=input_data, capture_output=True, text=True,
            timeout=5, cwd=TEMP_COMPILE_DIR
        )
        elapsed = time.time() - start
        if result.returncode == 0:
            return True, result.stdout.strip(), None, round(elapsed, 4)
        return False, None, f"Runtime error: {result.stderr.strip()}", None
    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None

def compare_outputs(actual, expected):
    return [l.strip() for l in actual.strip().split('\n')] == \
           [l.strip() for l in expected.strip().split('\n')]

def run_lizard(java_file_path):
    """
    Run lizard on a Java file.
    Returns dict with function_count, avg_ccn, avg_nloc, total_nloc or -1 on failure.
    """
    try:
        result = subprocess.run(
            ['lizard', java_file_path],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode != 0:
            return None

        lines = result.stdout.strip().split('\n')
        functions = []

        for line in lines:
            if '@' in line and not line.startswith('=') and not line.startswith('-') and 'NLOC' not in line:
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        functions.append({
                            'nloc':   int(parts[0]),
                            'ccn':    int(parts[1]),
                            'token':  int(parts[2]),
                            'param':  int(parts[3]),
                            'length': int(parts[4]),
                        })
                    except (ValueError, IndexError):
                        continue

        if not functions:
            return None  # lizard returned -1 / no functions

        return {
            'function_count':    len(functions),
            'total_nloc':        sum(f['nloc'] for f in functions),
            'avg_nloc':          sum(f['nloc'] for f in functions) / len(functions),
            'avg_ccn':           sum(f['ccn'] for f in functions) / len(functions),
            'max_ccn':           max(f['ccn'] for f in functions),
            'total_ccn':         sum(f['ccn'] for f in functions),
        }
    except subprocess.TimeoutExpired:
        return None
    except Exception:
        return None

def test_single_file(java_file_path, problem_id):
    result = {
        'compilation': {'success': False, 'error': None},
        'execution':   {'success': False, 'error': None, 'time_seconds': None},
        'test_validation': {'success': False, 'error': None},
        'overall_status': 'failed'
    }

    ok, err, cls = compile_java(java_file_path)
    result['compilation']['success'] = ok
    result['compilation']['error']   = err

    if not ok:
        clean_temp()
        return result

    inp_file = os.path.join(CODENET_DIR, problem_id, 'input.txt')
    out_file = os.path.join(CODENET_DIR, problem_id, 'output.txt')

    if not os.path.exists(inp_file) or not os.path.exists(out_file):
        result['execution']['error'] = 'input.txt or output.txt not found'
        clean_temp()
        return result

    with open(inp_file, 'r', encoding='utf-8') as f:
        test_input = f.read()
    with open(out_file, 'r', encoding='utf-8') as f:
        expected = f.read()

    ok, actual, err, elapsed = run_java_with_input(cls, test_input)
    result['execution']['success']      = ok
    result['execution']['error']        = err
    result['execution']['time_seconds'] = elapsed

    if not ok:
        clean_temp()
        return result

    match = compare_outputs(actual, expected)
    result['test_validation']['success'] = match
    result['test_validation']['error']   = None if match else \
        f"expected: {expected[:120]} | got: {actual[:120]}"

    if match:
        result['overall_status'] = 'passed'

    clean_temp()
    return result

# ============================================================
# MAIN
# ============================================================

def main():
    print("\n" + "="*80)
    print("RQ5 STRATEGY 1 — EVALUATION (Functional + Lizard)")
    print("="*80)
    print(f"\n>> Input:   {S1_CLEANED_DIR}")
    print(f">> Output:  {OUTPUT_FILE}")

    if not os.path.exists(LIBCOBJ_JAR):
        print(f"❌ libcobj.jar not found at {LIBCOBJ_JAR} — install opensourcecobol4j first")
        return

    problem_dirs = sorted([
        d for d in os.listdir(S1_CLEANED_DIR)
        if os.path.isdir(os.path.join(S1_CLEANED_DIR, d))
    ])
    print(f">> {len(problem_dirs)} programs to evaluate\n")

    all_results = []
    stats = {
        'total': 0,
        'compiled': 0,
        'executed': 0,
        'passed': 0,
        'compilation_failed': 0,
        'execution_failed': 0,
        'output_mismatch': 0,
        'lizard_valid': 0,
        'lizard_invalid': 0,
    }

    for pid in tqdm(problem_dirs, desc="Evaluating"):
        pid_dir    = os.path.join(S1_CLEANED_DIR, pid)
        java_files = sorted([f for f in os.listdir(pid_dir) if f.endswith('.java')])

        if not java_files:
            continue

        java_filename = java_files[0]
        java_path     = os.path.join(pid_dir, java_filename)

        stats['total'] += 1

        # Functional test
        test_result = test_single_file(java_path, pid)

        if test_result['compilation']['success']:
            stats['compiled'] += 1
        else:
            stats['compilation_failed'] += 1

        if test_result['execution']['success']:
            stats['executed'] += 1
        elif test_result['compilation']['success']:
            stats['execution_failed'] += 1

        if test_result['overall_status'] == 'passed':
            stats['passed'] += 1
        elif test_result['execution']['success']:
            stats['output_mismatch'] += 1

        # Lizard analysis
        lizard = run_lizard(java_path)
        if lizard:
            stats['lizard_valid'] += 1
        else:
            stats['lizard_invalid'] += 1

        all_results.append({
            'problem_id':   pid,
            'java_file':    java_filename,
            'functional':   test_result,
            'lizard':       lizard,
        })

    # ── Aggregate Lizard stats ──
    valid_lizard = [r['lizard'] for r in all_results if r['lizard']]
    lizard_summary = {}
    if valid_lizard:
        lizard_summary = {
            'avg_function_count': sum(l['function_count'] for l in valid_lizard) / len(valid_lizard),
            'avg_ccn':            sum(l['avg_ccn'] for l in valid_lizard) / len(valid_lizard),
            'avg_nloc':           sum(l['avg_nloc'] for l in valid_lizard) / len(valid_lizard),
            'avg_total_nloc':     sum(l['total_nloc'] for l in valid_lizard) / len(valid_lizard),
            'max_ccn':            max(l['max_ccn'] for l in valid_lizard),
        }

    # ── Save ──
    output = {
        'strategy': 'S1 — Code only refactoring',
        'summary':  stats,
        'lizard_summary': lizard_summary,
        'detailed_results': all_results,
    }
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(output, f, indent=2)

    # ── Print summary ──
    total = stats['total']
    print(f"\n{'='*80}")
    print(f"FUNCTIONAL TESTING SUMMARY")
    print(f"{'='*80}")
    print(f"  Total files:          {total}")
    print(f"  Compiled:             {stats['compiled']}/{total} ({stats['compiled']/total*100:.1f}%)")
    print(f"  Executed:             {stats['executed']}/{total} ({stats['executed']/total*100:.1f}%)")
    print(f"  Passed:               {stats['passed']}/{total} ({stats['passed']/total*100:.1f}%)")
    print(f"  Compilation failed:   {stats['compilation_failed']}")
    print(f"  Execution failed:     {stats['execution_failed']}")
    print(f"  Output mismatch:      {stats['output_mismatch']}")

    print(f"\n{'='*80}")
    print(f"LIZARD SUMMARY")
    print(f"{'='*80}")
    print(f"  Valid analyses:       {stats['lizard_valid']}/{total}")
    print(f"  Invalid (skipped):    {stats['lizard_invalid']}")
    if lizard_summary:
        print(f"  Avg function count:   {lizard_summary['avg_function_count']:.1f}")
        print(f"  Avg CCN:              {lizard_summary['avg_ccn']:.2f}")
        print(f"  Avg NLOC/function:    {lizard_summary['avg_nloc']:.1f}")
        print(f"  Avg total NLOC:       {lizard_summary['avg_total_nloc']:.1f}")
        print(f"  Max CCN:              {lizard_summary['max_ccn']}")

    print(f"\n>> Results saved to: {OUTPUT_FILE}")
    print(">> Done!")

if __name__ == "__main__":
    main()